# Ноутбук 5. Наклонное зеркало: смена базиса + Грам–Шмидт
*(к уроку 10 курса)*

Задача: отразить точку в зеркале, которое стоит **наклонно**.

План (гениально ленивый):

1. перейти в систему координат, где зеркало — это ось x: `E.T @ r`
2. там отражение простое — перевернуть вторую координату: `TE @ ...`
3. вернуться в обычные координаты: `E @ ...`

Итого: **`r_new = E @ TE @ E.T @ r`** (читаем справа налево!).
Матрицу `E` (честные оси зеркала) строит наш Грам–Шмидт.

In [ ]:
import numpy as np
import numpy.linalg as la
import matplotlib.pyplot as plt

verySmallNumber = 1e-14

def gsBasis(A):                     # готовая функция из ноутбука 4
    B = np.array(A, dtype=np.float64)
    for j in range(B.shape[1]):
        for i in range(j):
            B[:, j] = B[:, j] - B[:, j] @ B[:, i] * B[:, i]
        if la.norm(B[:, j]) > verySmallNumber:
            B[:, j] = B[:, j] / la.norm(B[:, j])
        else:
            B[:, j] = np.zeros_like(B[:, j])
    return B

print("gsBasis загружена")

## 1. Строим матрицу отражения

In [ ]:
# "Кривой" базис медведя: первый столбец - направление зеркала
bearBasis = np.array([[1, -1],
                      [1.5, 2]], dtype=np.float64)

E = gsBasis(bearBasis)          # шаг 1: выпрямляем базис
print("E (столбцы - оси зеркала: e1 вдоль, e2 поперёк):")
print(np.round(E, 3))

TE = np.array([[1, 0],          # шаг 2: простое отражение "в мире зеркала"
               [0, -1]])

T = E @ TE @ E.T                # шаг 3: склеиваем цепочку в одну матрицу
print()
print("T = E @ TE @ E.T - машинка наклонного отражения:")
print(np.round(T, 3))

Почему `E.T`, а не обратная матрица `la.inv(E)`? Потому что `E` после
Грама–Шмидта **ортонормированная**, а у таких матриц обратная равна
транспонированной: `E⁻¹ = E.T`. Проверь:

In [ ]:
print("E.T @ E =")
print(np.round(E.T @ E, 10))        # единичная матрица - значит E.T это и есть обратная
print()
print("la.inv(E) =", )
print(np.round(la.inv(E), 3))
print("E.T =")
print(np.round(E.T, 3))             # совпадают!

## 2. Отражаем точку и смотрим всю цепочку по шагам

In [ ]:
r = np.array([1.0, 2.0])           # точка, которую отражаем

shag1 = E.T @ r                    # координаты точки В СИСТЕМЕ ЗЕРКАЛА
shag2 = TE @ shag1                 # отразили: вторая координата сменила знак
shag3 = E @ shag2                  # вернулись в обычные координаты

print("точка r:                     ", r)
print("шаг 1, E.T @ r:              ", np.round(shag1, 3), " <- (вдоль зеркала, поперёк)")
print("шаг 2, TE @ ...:             ", np.round(shag2, 3), " <- поперёк сменила знак")
print("шаг 3, E @ ...:              ", np.round(shag3, 3), " <- отражение готово")
print()
print("одним махом, T @ r:          ", np.round(T @ r, 3), " <- то же самое!")

## 3. Рисуем зеркало, точку и отражение

In [ ]:
def risuy_zerkalo(bearBasis, r):
    E = gsBasis(bearBasis)
    T = E @ np.array([[1, 0], [0, -1]]) @ E.T
    ro = T @ r

    plt.figure(figsize=(6.5, 6.5))
    ax = plt.gca()
    ax.axhline(0, color="#ccc"); ax.axvline(0, color="#ccc")
    e1 = E[:, 0]
    ax.plot([-4*e1[0], 4*e1[0]], [-4*e1[1], 4*e1[1]],
            color="tab:green", lw=8, alpha=0.25, label="зеркало")
    ax.plot(*r, "o", color="tab:blue", ms=11)
    ax.text(r[0]+0.15, r[1]+0.15, "r", color="tab:blue", fontsize=13, weight="bold")
    ax.plot(*ro, "o", color="tab:pink", ms=11)
    ax.text(ro[0]+0.15, ro[1]+0.15, "отражение", color="tab:pink", fontsize=12, weight="bold")
    ax.plot([r[0], ro[0]], [r[1], ro[1]], "--", color="#999")
    ax.set_xlim(-4, 4); ax.set_ylim(-4, 4); ax.set_aspect("equal")
    ax.grid(True, alpha=0.3); ax.legend()
    plt.show()

risuy_zerkalo(bearBasis, np.array([1.0, 2.0]))

**Поиграйся:** меняй точку и базис зеркала, запускай. Пунктир всегда
пересекает зеркало под прямым углом, а точки — на равном расстоянии от него.

---
# 🏋️ Задания

In [ ]:
# Помощник для проверки заданий. Просто запусти эту ячейку (Shift+Enter)
# и не думай пока о том, как она устроена, - она будет проверять твои ответы.
import numpy as np

def prover(nazvanie, tvoy_otvet, pravilnyy):
    """Сравнивает твой ответ с правильным и печатает результат."""
    if tvoy_otvet is ... or tvoy_otvet is None:
        print("⏳", nazvanie, "- ещё не решено (замени ... на свой ответ)")
        return
    try:
        if np.allclose(np.array(tvoy_otvet, dtype=np.float64),
                       np.array(pravilnyy, dtype=np.float64), atol=1e-6):
            print("✅", nazvanie, "- ПРАВИЛЬНО!")
        else:
            print("❌", nazvanie, "- пока неверно. Твой ответ:", tvoy_otvet)
    except Exception as e:
        print("❌", nazvanie, "- ответ не получилось сравнить:", e)

print("Проверяльщик готов!")

### Задание 1. Зеркало по диагонали

Зеркало вдоль вектора `[1, 1]` (bearBasis: первый столбец `[1, 1]`, второй `[0, 1]`).
Куда отразится точка `[3, 0]`? Сначала **догадайся по картинке в голове**
(зеркало — диагональ под 45°!), впиши ответ, потом проверь кодом.

In [ ]:
moy_otvet = ...        # куда попадёт [3, 0]? вектор из двух чисел

prover("отражение в диагонали", moy_otvet, [0, 3])

In [ ]:
# А теперь проверь себя кодом - построй T и посчитай:
bearBasis2 = np.array([[1.0, 0.0],
                       [1.0, 1.0]])
E2 = ...               # примени gsBasis
T2 = ...               # собери цепочку E2 @ TE @ E2.T

# когда заполнишь, раскомментируй:
# print("T2 @ [3, 0] =", T2 @ np.array([3.0, 0.0]))
# risuy_zerkalo(bearBasis2, np.array([3.0, 0.0]))

### Задание 2. Свойства отражения (проверь кодом в свободной ячейке)

1. `T @ (T @ r)` — что получится и почему?
2. Отрази точку, лежащую **прямо на зеркале** (например, `2 * E[:, 0]`). Что вышло?
3. Сравни `la.norm(r)` и `la.norm(T @ r)`.

In [ ]:
# свободная ячейка для проверки свойств

<details>
<summary>👉 Решения</summary>

<pre>
Задание 1: [0, 3] - зеркало-диагональ меняет координаты местами.
  E2 = gsBasis(bearBasis2)
  T2 = E2 @ np.array([[1, 0], [0, -1]]) @ E2.T

Задание 2:
  1) исходная точка r: два отражения = вернуться назад;
  2) точка на зеркале отражается сама в себя;
  3) длины равны: зеркало не растягивает.
</pre>
</details>

---
Финал — **ноутбук 6**: тренажёр, который придумывает задачи за тебя.